Handling of Bulk Data 

In [4]:
import pandas as pd
import numpy as np

In [5]:
df=pd.read_csv("data/raw/netflix_titles.csv")
df

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...
...,...,...,...,...,...,...,...,...,...,...,...,...
8802,s8803,Movie,Zodiac,David Fincher,"Mark Ruffalo, Jake Gyllenhaal, Robert Downey J...",United States,"November 20, 2019",2007,R,158 min,"Cult Movies, Dramas, Thrillers","A political cartoonist, a crime reporter and a..."
8803,s8804,TV Show,Zombie Dumb,NaN,NaN,NaN,"July 1, 2019",2018,TV-Y7,2 Seasons,"Kids' TV, Korean TV Shows, TV Comedies","While living alone in a spooky town, a young g..."
8804,s8805,Movie,Zombieland,Ruben Fleischer,"Jesse Eisenberg, Woody Harrelson, Emma Stone, ...",United States,"November 1, 2019",2009,R,88 min,"Comedies, Horror Movies",Looking to survive in a world taken over by zo...
8805,s8806,Movie,Zoom,Peter Hewitt,"Tim Allen, Courteney Cox, Chevy Chase, Kate Ma...",United States,"January 11, 2020",2006,PG,88 min,"Children & Family Movies, Comedies","Dragged from civilian life, a former superhero..."


In [6]:
print(df.columns)
print("\n",df.shape)
print("\n",df.isnull().sum())

Index(['show_id', 'type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description'],
      dtype='object')

 (8807, 12)

 show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64


In [7]:
# Content table 
content=df[['title','type','release_year','description']].copy()
content.head()

,title,type,release_year,description
0,Dick Johnson Is Dead,Movie,2020,"As her father nears the end of his life, filmm..."
1,Blood & Water,TV Show,2021,"After crossing paths at a party, a Cape Town t..."
2,Ganglands,TV Show,2021,To protect his family from a powerful drug lor...
3,Jailbirds New Orleans,TV Show,2021,"Feuds, flirtations and toilet talk go down amo..."
4,Kota Factory,TV Show,2021,In a city of coaching centers known to train I...


In [8]:
content.insert(
    0,
    'content_id',
    range(1, len(content)+1)
)

In [9]:
df['director'].dropna().head(20)

0                                   Kirsten Johnson
2                                   Julien Leclercq
5                                     Mike Flanagan
6                     Robert Cullen, José Luis Ucha
7                                      Haile Gerima
8                                   Andy Devonshire
9                                    Theodore Melfi
11                                Kongkiat Komesiri
12                              Christian Schwochow
13                                    Bruno Garotti
16    Pedro de Echave García, Pablo Azorín Williams
18                                       Adam Salky
20                                  Olivier Megaton
22                                   K.S. Ravikumar
23                          Alex Woo, Stanley Moore
24                                       S. Shankar
26                                      Rajiv Menon
27                                     Dennis Dugan
28                                    Scott Stewart
29          

In [10]:
#directors table 
directors = df[['director']].dropna().copy()
directors['director'] = directors['director'].str.split(',')
directors=directors.explode('director')
directors['director'] = directors['director'].str.strip()
directors=directors[['director']].drop_duplicates()

In [11]:
directors.insert(
    0,
    'director_id',
    range(1, len(directors)+1)
)

In [12]:
# directs table 
directs = df[['title','director']].dropna().copy()
directs['director']=directs['director'].str.split(',')
directs = directs.explode('director')
directs['director'] = directs['director'].str.strip()

In [13]:
directs = directs.merge(
    content[['content_id', 'title']],
    on='title'
)
directs = directs.merge(
    directors[['director_id', 'director']],
    on='director'
)

In [14]:
directs = directs[['director_id','content_id']]
directs = directs.drop_duplicates()
directs.head()

,director_id,content_id
0,1,1
1,2,3
2,3,6
3,4,7
4,5,7


In [15]:
#actors table 
actors = df[['cast']].dropna().copy()
actors['cast'] = actors['cast'].str.split(',')
actors = actors.explode('cast')
actors['cast'] = actors['cast'].str.strip()
actors = actors[['cast']].drop_duplicates()
actors = actors.rename(columns={'cast': 'name'})

In [16]:
actors.insert(
    0,
    'actor_id',
    range(1, len(actors)+1)
)

In [17]:
# acts_in table 
acts_in = df[['title', 'cast']].dropna().copy()
acts_in['cast'] = acts_in['cast'].str.split(',')
acts_in = acts_in.explode('cast')
acts_in['cast'] = acts_in['cast'].str.strip()

In [18]:
acts_in = acts_in.merge(
    content[['content_id', 'title']],
    on='title'
)
acts_in = acts_in.merge(
    actors[['actor_id', 'name']],
    left_on='cast',
    right_on='name'
)

In [19]:
acts_in = acts_in[
    ['actor_id', 'content_id']
].drop_duplicates()
acts_in.head()

,actor_id,content_id
0,1,2
1,2,2
2,3,2
3,4,2
4,5,2


In [20]:
#genre table 
genres = df[['listed_in']].copy()
genres['listed_in'] = genres['listed_in'].str.split(',')
genres = genres.explode('listed_in')
genres['listed_in'] = genres['listed_in'].str.strip()
genres = genres[['listed_in']].drop_duplicates()
genres = genres.rename(
    columns={'listed_in':'name'}
)

In [21]:
genres.insert(
    0,
    'genre_id',
    range(1, len(genres)+1)
)

In [22]:
genres.head()

,genre_id,name
0,1,Documentaries
1,2,International TV Shows
1,3,TV Dramas
1,4,TV Mysteries
2,5,Crime TV Shows


In [23]:
#has genre table 
has_genre = df[['title','listed_in']].copy()
has_genre['listed_in'] = has_genre['listed_in'].str.split(',')
has_genre = has_genre.explode('listed_in')
has_genre['listed_in'] = has_genre['listed_in'].str.strip()

In [24]:
has_genre = has_genre.merge(
    content[['content_id','title']],
    on='title'
)
has_genre = has_genre.merge(
    genres[['genre_id','name']],
    left_on='listed_in',
    right_on='name'
)

In [25]:
has_genre = has_genre[
    ['genre_id','content_id']
].drop_duplicates()

In [26]:
has_genre.head()

,genre_id,content_id
0,1,1
1,2,2
2,3,2
3,4,2
4,5,3


In [27]:
#converting to csv
content.to_csv('data/processed/tbl_content.csv', index=False)
directors.to_csv('data/processed/tbl_directors.csv', index=False)
directs.to_csv('data/processed/tbl_directs.csv', index=False)
actors.to_csv('data/processed/tbl_actors.csv', index=False)
acts_in.to_csv('data/processed/tbl_acts_in.csv', index=False)
genres.to_csv('data/processed/tbl_genre.csv', index=False)
has_genre.to_csv('data/processed/tbl_has_genre.csv', index=False)


In [28]:
print(content.shape)
print(directors.shape)
print(directs.shape)
print(actors.shape)
print(acts_in.shape)
print(genres.shape)
print(has_genre.shape)

(8807, 5)
(4993, 2)
(6977, 2)
(36439, 2)
(64124, 2)
(42, 2)
(19323, 2)
